# Publication-Quality Plots

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

## ALL FEATURES (CLASSIFICATION)

In [2]:
dir = 'output/All Features'

# LMS
lms_imp = pd.read_csv("lms_rf_quenching.csv")

# lms_imp = lms[['imp_feature', 'imp_importance', 'imp_std']]
# lms_metrics = lms[['Accuracy', 'F1 Score', 'Recall', 'Precision', 'ROC AUC (Train)', 'ROC AUC (Test)']]

# # HMS
hms_imp = pd.read_csv("hms_rf_quenching.csv")

# hms = pd.read_csv(f"{dir}/HMS/metrics.csv")
# hms_imp = hms[['imp_feature', 'imp_importance', 'imp_std']]
# hms_metrics = hms[['Accuracy', 'F1 Score', 'Recall', 'Precision', 'ROC AUC (Train)', 'ROC AUC (Test)']]

# # LMC
lmc_imp = pd.read_csv("lmc_rf_quenching.csv")

# lmc = pd.read_csv(f"{dir}/LMC/metrics.csv")
# lmc_imp = lmc[['imp_feature', 'imp_importance', 'imp_std']]
# lmc_metrics = lmc[['Accuracy', 'F1 Score', 'Recall', 'Precision', 'ROC AUC (Train)', 'ROC AUC (Test)']]

# # HMC
hmc_imp = pd.read_csv("hmc_rf_quenching.csv")

# hmc = pd.read_csv(f"{dir}/HMC/metrics.csv")
# hmc_imp = hmc[['imp_feature', 'imp_importance', 'imp_std']]
# hmc_metrics = hmc[['Accuracy', 'F1 Score', 'Recall', 'Precision', 'ROC AUC (Train)', 'ROC AUC (Test)']]

# hmc_imp

In [3]:
def four_plot(imps, metrics=None, save_path=None, pie_xy=(0.66, 0.55)):
    """
    4-panel (2×2) publication-quality plot of feature importances,
    vertical bars with black outlines, LaTeX feature names, y-axis on every panel,
    color-coded local vs. global parameters, inset pie charts,
    a publication-style legend in Times New Roman, and black subplot borders.

    Parameters
    ----------
    imps : list of pd.DataFrame
        4 DataFrames (LMS, HMS, LMC, HMC) with
        'imp_feature', 'imp_importance', 'imp_std'.
    metrics : list of pd.DataFrame, optional
        (unused here; reserved for future extensions)
    save_path : str, optional
        Path to save figure (PDF/SVG).
    pie_xy : tuple of floats
        (x, y) position within each axes (in axes fraction) to place
        the inset pie chart.
    """
    # LaTeX + Times New Roman via newtx
    plt.rcParams.update({
        "text.usetex": True,
        "font.family": "serif",
        "font.serif": ["Times New Roman"],
        "text.latex.preamble": r"\usepackage{newtxtext,newtxmath}",
    })
    import seaborn as sns
    sns.set_style("whitegrid")
    sns.set_context("paper", font_scale=1.4)

    local_features = {'r_bin', 'gas_mass_bin', 's_mass_bin'}
    local_color = 'tab:blue'
    global_color = 'tab:purple'

    latex_labels = {
        'r_bin':        r'$r_{\rm bin}$',
        's_mass_bin':   r'$M_{\odot,\;\rm bin}$',
        'BHMass':       r'$M_{\rm BH}$',
        'Mhalo':        r'$M_{\rm h}$',
        'BH_cum_tot':   r'$\Sigma_{\rm EQ}$',
        'bulge':        r'$F(G,\;M_{20})$',
        'overdens':     r'$\delta_{5}$',
        # 'BH_AGN': r'$\sum E_{\rm{AGN}}$'
    }
    # Order: [LMS, LMC, HMS, HMC] for correct subplot placement
    names = ['Low Mass Satellites', 'Low Mass Centrals', 'High Mass Satellites', 'High Mass Centrals']
    # Rearranged imps for subplot order: [LMS, LMC, HMS, HMC]
    imps = [imps[0], imps[2], imps[1], imps[3]]

    max_imp = max((df['Mean Importance'] + df['Std Dev']).max() for df in imps) * 1.1

    fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharey=True)
    axes = axes.flatten()

    for i, (df, ax, label) in enumerate(zip(imps, axes, names)):
        df = df.sort_values('Mean Importance', ascending=False).copy()
        df['orig_feature'] = df['Feature']
        df['Feature'] = df['Feature'].map(lambda x: latex_labels.get(x, x))

        colors = [local_color if feat in local_features else global_color for feat in df['orig_feature']]
        ax.bar(
            df['Feature'], df['Mean Importance'], yerr=df['Std Dev'],
            capsize=4, color=colors, edgecolor='black', linewidth=0.8, alpha=0.9
        )

        # Black border around subplot
        for spine in ax.spines.values():
            spine.set_edgecolor('black')
            spine.set_linewidth(0.8)

        ax.set_title(label, fontsize=20)
        ax.set_ylim(0, max_imp)
        ax.grid(axis='y', linestyle='--', alpha=0.5)
        ax.tick_params(axis='x', labelrotation=45, labelsize=14)
        ax.tick_params(axis='y', labelsize=14)
        ax.set_ylabel(r'Relative Importance', fontsize=18)

        # Inset pie chart (no percentages)
        x0, y0 = pie_xy
        axins = ax.inset_axes([x0, y0, 0.45, 0.45], transform=ax.transAxes)
        local_sum = df.loc[df['orig_feature'].isin(local_features), 'Mean Importance'].sum()
        global_sum = df.loc[~df['orig_feature'].isin(local_features), 'Mean Importance'].sum()
        axins.pie([local_sum, global_sum], colors=[local_color, global_color], startangle=90,
                  wedgeprops={'alpha':0.7, 'linewidth':0.5, 'edgecolor':'black'})
        axins.set_aspect('equal')

    # Legend
    handles = [
        plt.Line2D([0], [0], marker='s', color=local_color, lw=0, markersize=10, label='Local'),
        plt.Line2D([0], [0], marker='s', color=global_color, lw=0, markersize=10, label='Global')
    ]
    legend = fig.legend(handles=handles, loc='upper left', bbox_to_anchor=(0.02, 0.98),
                        frameon=True, edgecolor='black', fontsize=16)
    legend.get_frame().set_linewidth(0.8)

    # Remove the overall title
    plt.tight_layout(rect=[0, 0.02, 1, 0.98])

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

In [4]:
imps = [lms_imp, hms_imp, lmc_imp, hmc_imp]
# metrics = [lms_metrics, lms_metrics, lms_metrics, lms_metrics]
four_plot(imps)

RuntimeError: Failed to process string with tex because latex could not be found

Error in callback <function _draw_all_if_interactive at 0x15199b331620> (for post_execute), with arguments args (),kwargs {}:


RuntimeError: Failed to process string with tex because latex could not be found

RuntimeError: Failed to process string with tex because latex could not be found

<Figure size 1600x1200 with 4 Axes>